# Predicting Stellar Class — improved baseline

This notebook uses colour features, proper stratified cross-validation, and a probability-based ensemble. Run cells from top to bottom.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, log_loss, confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

import lightgbm as lgb
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

SEED = 42
N_SPLITS = 5
OUTPUT_DIR = Path('data')
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
# Set this if your Kaggle files are in another location.
DATA_DIR = Path.home() / '.cache/kagglehub/competitions/playground-series-s6e6'

# Alternative: uncomment the next two lines if the files have not been downloaded yet.
# import kagglehub
# DATA_DIR = Path(kagglehub.competition_download('playground-series-s6e6'))

train_df = pd.read_csv(DATA_DIR / 'train.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
print(train_df.shape, test_df.shape)
display(train_df['class'].value_counts(normalize=True).rename('share'))

(577347, 12) (247435, 11)


class
GALAXY    0.653818
QSO       0.202899
STAR      0.143283
Name: share, dtype: float64

## Feature engineering

Astronomical magnitudes are most informative as *colour indices* (differences between bands). We deliberately exclude `id`: it identifies rows, not stars.

In [3]:
TARGET = 'class'
CATEGORICAL_COLUMNS = ['spectral_type', 'galaxy_population']

def make_features(frame: pd.DataFrame) -> pd.DataFrame:
    X = frame.drop(columns=['id', TARGET], errors='ignore').copy()
    # Adjacent colours are physically meaningful and easy for tree models to use.
    for left, right in [('u', 'g'), ('g', 'r'), ('r', 'i'), ('i', 'z')]:
        X[f'{left}_minus_{right}'] = X[left] - X[right]
    # Right ascension is circular: 0 and 360 degrees describe neighbouring directions.
    alpha_radians = np.deg2rad(X['alpha'])
    X['alpha_sin'] = np.sin(alpha_radians)
    X['alpha_cos'] = np.cos(alpha_radians)
    return X

X = make_features(train_df)
X_test = make_features(test_df)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_df[TARGET])
print(X.shape, X_test.shape, label_encoder.classes_)

(577347, 16) (247435, 16) ['GALAXY' 'QSO' 'STAR']


In [4]:
# These are strong starting settings. Tune only after this baseline is reproducible.
LGB_PARAMS = dict(
    objective='multiclass', n_estimators=1500, learning_rate=0.03,
    num_leaves=96, max_depth=-1, min_child_samples=40,
    subsample=0.85, colsample_bytree=0.9, reg_lambda=1.0,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
CAT_PARAMS = dict(
    loss_function='MultiClass', iterations=1000, learning_rate=0.05, depth=8,
    l2_leaf_reg=4.0, random_strength=1.0, random_seed=SEED,
    verbose=False, allow_writing_files=False, thread_count=-1,
)
XGB_PARAMS = dict(
    objective='multi:softprob', num_class=len(label_encoder.classes_),
    n_estimators=1000, learning_rate=0.03, max_depth=9, min_child_weight=8,
    subsample=0.9, colsample_bytree=0.85, reg_lambda=0.2,
    eval_metric='mlogloss', random_state=SEED, n_jobs=-1,
    early_stopping_rounds=100,
)

In [5]:
# Important: the category encoder is fitted on each training fold only.
# CatBoost receives the original category strings; LightGBM/XGBoost use fold-safe ordinal values.
n_classes = len(label_encoder.classes_)
oof = {name: np.zeros((len(X), n_classes)) for name in ['lgb', 'cat', 'xgb']}
test_proba = {name: np.zeros((len(X_test), n_classes)) for name in oof}
splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

for fold, (train_idx, valid_idx) in enumerate(splitter.split(X, y), start=1):
    print(f'Fold {fold}/{N_SPLITS}')
    X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
    y_train, y_valid = y[train_idx], y[valid_idx]

    # CatBoost: use raw categorical values, its native and preferred approach.
    cat_train, cat_valid, cat_test = X_train.copy(), X_valid.copy(), X_test.copy()
    for col in CATEGORICAL_COLUMNS:
        cat_train[col] = cat_train[col].fillna('missing').astype(str)
        cat_valid[col] = cat_valid[col].fillna('missing').astype(str)
        cat_test[col] = cat_test[col].fillna('missing').astype(str)
    cat_model = CatBoostClassifier(**CAT_PARAMS)
    cat_model.fit(cat_train, y_train, cat_features=CATEGORICAL_COLUMNS,
                  eval_set=(cat_valid, y_valid), early_stopping_rounds=100, verbose=False)
    oof['cat'][valid_idx] = cat_model.predict_proba(cat_valid)
    test_proba['cat'] += cat_model.predict_proba(cat_test) / N_SPLITS

    # LightGBM/XGBoost: encoded without looking at validation or test categories.
    encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1,
                             encoded_missing_value=-1)
    encoded_train, encoded_valid, encoded_test = X_train.copy(), X_valid.copy(), X_test.copy()
    encoded_train[CATEGORICAL_COLUMNS] = encoder.fit_transform(encoded_train[CATEGORICAL_COLUMNS])
    encoded_valid[CATEGORICAL_COLUMNS] = encoder.transform(encoded_valid[CATEGORICAL_COLUMNS])
    encoded_test[CATEGORICAL_COLUMNS] = encoder.transform(encoded_test[CATEGORICAL_COLUMNS])

    lgb_model = lgb.LGBMClassifier(**LGB_PARAMS)
    lgb_model.fit(encoded_train, y_train, eval_set=[(encoded_valid, y_valid)],
                  callbacks=[lgb.early_stopping(100, verbose=False)])
    oof['lgb'][valid_idx] = lgb_model.predict_proba(encoded_valid)
    test_proba['lgb'] += lgb_model.predict_proba(encoded_test) / N_SPLITS

    xgb_model = XGBClassifier(**XGB_PARAMS)
    xgb_model.fit(encoded_train, y_train, eval_set=[(encoded_valid, y_valid)], verbose=False)
    oof['xgb'][valid_idx] = xgb_model.predict_proba(encoded_valid)
    test_proba['xgb'] += xgb_model.predict_proba(encoded_test) / N_SPLITS

Fold 1/5


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2/5


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 3/5


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4/5


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5/5


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


In [6]:
# Compare models using both the leaderboard-style accuracy and balanced accuracy.
for name, probabilities in oof.items():
    prediction = probabilities.argmax(axis=1)
    print(f'{name:>3} | accuracy={accuracy_score(y, prediction):.5f} | '
          f'balanced_accuracy={balanced_accuracy_score(y, prediction):.5f} | '
          f'logloss={log_loss(y, probabilities):.5f}')

best_name = max(oof, key=lambda name: accuracy_score(y, oof[name].argmax(axis=1)))
print(f'Best individual model by OOF accuracy: {best_name}')
print(confusion_matrix(y, oof[best_name].argmax(axis=1)))

lgb | accuracy=0.96864 | balanced_accuracy=0.95730 | logloss=0.08777
cat | accuracy=0.96512 | balanced_accuracy=0.95134 | logloss=0.09627
xgb | accuracy=0.96860 | balanced_accuracy=0.95709 | logloss=0.08761
Best individual model by OOF accuracy: lgb
[[369416   3324   4740]
 [  3134 113025    984]
 [  5428    494  76802]]


/home/rpsingh/Workspace/machine-learning/kaggle-practices/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:304: UserWarning: The y_prob values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


In [7]:
# Find soft-voting weights on OOF probabilities. This is valid stacking: no hard labels.
names = ['lgb', 'cat', 'xgb']
best_score, best_weights = -np.inf, None
for w_lgb in np.arange(0, 1.01, 0.05):
    for w_cat in np.arange(0, 1.01 - w_lgb, 0.05):
        w_xgb = 1.0 - w_lgb - w_cat
        if w_xgb < -1e-9:
            continue
        weights = np.array([w_lgb, w_cat, max(w_xgb, 0)])
        blended = sum(weight * oof[name] for weight, name in zip(weights, names))
        score = accuracy_score(y, blended.argmax(axis=1))
        if score > best_score:
            best_score, best_weights = score, weights

print('Best OOF ensemble accuracy:', round(best_score, 5))
print(dict(zip(names, best_weights.round(2))))

Best OOF ensemble accuracy: 0.96893
{'lgb': np.float64(0.5), 'cat': np.float64(0.0), 'xgb': np.float64(0.5)}


In [8]:
# Generate the final Kaggle file.
final_proba = sum(weight * test_proba[name] for weight, name in zip(best_weights, names))
final_class = label_encoder.inverse_transform(final_proba.argmax(axis=1))
submission = pd.DataFrame({'id': test_df['id'], 'class': final_class})
submission_path = OUTPUT_DIR / 'stellar_soft_voting_submission.csv'
submission.to_csv(submission_path, index=False)

assert submission['id'].equals(test_df['id'])
assert len(submission) == len(test_df)
display(submission.head())
print(f'Saved: {submission_path.resolve()}')

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY


Saved: /home/rpsingh/Workspace/machine-learning/kaggle-practices/Predicting-Stellar-Class/data/stellar_soft_voting_submission.csv
